# Earth2Studio grids

A grid object describes spatial geometry without containing weather data. This tutorial introduces the protocol, built-in definitions, registry, inference, and selection hooks.

## Define a grid object

A grid object structurally implements the `GridDefinition` protocol. It does not inherit from a grid base class. The interface makes dimension order, shape, topology, coordinate generation, metadata, and identity explicit.

```python
class GridDefinition(Protocol):
    @property
    def dims(self) -> tuple[str, ...]: ...

    @property
    def shape(self) -> tuple[int, ...]: ...

    @property
    def topology(self) -> GridTopology: ...

    def index_coordinates(self) -> xr.Coordinates: ...

    def geographic_coordinates(self, indexes) -> xr.Coordinates: ...

    def to_metadata(self) -> dict[str, object]: ...

    def fingerprint(self) -> str: ...
```

The protocol also includes `crs`, `subset_indexers()`, and `cell_bounds()`. A grid may return `None` for geometry it cannot provide.

In [ ]:
import numpy as np
import xarray as xr

import earth2studio as e2s

## Use a known grid

The registry maps stable names and aliases to complete definitions. `resolve_grid()` returns the definition itself, so its interface is discoverable.

In [ ]:
hrrr = e2s.resolve_grid("hrrr")
{
    "type": type(hrrr).__name__,
    "dims": hrrr.dims,
    "shape": hrrr.shape,
    "topology": hrrr.topology,
    "crs": hrrr.crs.name,
}

## Register a projected grid

Choose one of the concrete definitions. A projected grid requires ordered `y` and `x` coordinates plus any CRS accepted by PyProj. Registration adds identity; the definition supplies behavior.

In [ ]:
regional = e2s.ProjectedGrid(
    y=np.arange(3) * 3_000.0,
    x=np.arange(4) * 3_000.0,
    coordinate_reference_system=(
        "+proj=lcc +lat_1=30 +lat_2=60 +lat_0=38 +lon_0=-97 "
        "+datum=WGS84 +units=m +type=crs"
    ),
)
e2s.register_grid("tutorial-lcc", regional)

A grid stores its compact native indexes. Geographic coordinates are generated only when requested.

In [ ]:
indexes = regional.index_coordinates()
located = regional.geographic_coordinates(
    {dimension: np.asarray(indexes[dimension]) for dimension in regional.dims}
)
regional.shape, located["lat"].shape

## Infer ordinary Xarray grids

Registration is optional. Separate one-dimensional latitude and longitude coordinates define a rectilinear grid.

In [ ]:
latlon = xr.DataArray(
    np.zeros((2, 3)),
    dims=("lat", "lon"),
    coords={"lat": [40.0, 39.0], "lon": [250.0, 251.0, 252.0]},
)
type(e2s.infer_grid(latlon)).__name__

Two-dimensional `lat` and `lon` coordinates on `y, x` define a curvilinear grid. For arbitrary locations, `x` is the ordered index and `lat` and `lon` are auxiliary coordinates on it.

In [ ]:
points = xr.Dataset(
    coords={
        "x": np.arange(3),
        "lat": ("x", [35.2, 40.8, 51.0]),
        "lon": ("x", [-97.4, -74.0, 0.1]),
    }
)
point_grid = e2s.infer_grid(points)
point_grid.dims, point_grid.topology, point_grid.index_coordinates()

## Select without regridding

Geographic bounds are translated by the grid into normal Xarray indexers. Field data remains outside the grid implementation.

In [ ]:
grid = e2s.infer_grid(latlon)
indexers = grid.subset_indexers(
    latlon.coords, bounds=(-110, 38, -90, 41)
)
subset = latlon.isel(indexers)
subset.shape, subset.coords

## Regridding readiness

A regridder can accept any source and target that implement `GridDefinition`. It can use `topology`, `crs`, geographic centers, and optional cell bounds to choose a supported engine and method. `fingerprint()` provides a stable key for caching the resulting weights. Field data stays outside both grid objects.

```python
source_grid = e2s.infer_grid(source_array)
target_grid = e2s.resolve_grid("hrrr")
plan = regridder.plan(source_grid, target_grid, method="linear")
result = plan(source_array)
```

The regridder owns capability dispatch, weight construction, and application. Unsupported topology pairs or methods should raise before processing data.